In [1]:
!pip install -q open3d plyfile scikit-learn pandas numpy matplotlib seaborn tqdm
print('Dependencies ready.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 19.5 MB/s eta 0:00:00
Dependencies ready.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# --- Dataset location -------------------------------------------------------
BASE_DIR   = "/content/drive/MyDrive/segmentation_jetcobot_internship"
PLANT_NAME = "Ribes_04"

# --- Dataset directories ----------------------------------------------------
GT_DIR       = os.path.join(BASE_DIR, "gt_reconstruction")
NOISY_DIR    = os.path.join(BASE_DIR, "noisy_reconstruction")
DEGRADED_DIR = os.path.join(NOISY_DIR, "degraded")

# --- Ground-truth files ----------------------------------------------------
PLY_PATH        = os.path.join(GT_DIR, f"{PLANT_NAME}.ply")
SEMANTIC_PATH   = os.path.join(GT_DIR, f"{PLANT_NAME}_SemanticLabels.txt")
INSTANCE_PATH   = os.path.join(GT_DIR, f"{PLANT_NAME}_InstanceLabels.txt")
CONFIDENCE_PATH = os.path.join(GT_DIR, f"{PLANT_NAME}_Confidence.txt")

# --- Manifest ----------------------------------------------------------------
MANIFEST_PATH = os.path.join(DEGRADED_DIR, "manifest.csv")

# --- Output location --------------------------------------------------------
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs", PLANT_NAME)

ROBUST_DIR  = os.path.join(OUTPUT_DIR, "robustness_eval_dgcnn")
FIG_DIR     = os.path.join(ROBUST_DIR, "figures")
METRICS_DIR = os.path.join(ROBUST_DIR, "tables")
CM_DIR      = os.path.join(METRICS_DIR, "confusion_matrices")
REPORT_DIR  = os.path.join(ROBUST_DIR, "report")
CKPT_DIR   = os.path.join(ROBUST_DIR, "checkpoints")

# --- Create ALL directories -------------------------------------------------
ALL_DIRS = [
    # Dataset
    BASE_DIR,
    GT_DIR,
    NOISY_DIR,
    DEGRADED_DIR,

    # Output
    os.path.join(BASE_DIR, "outputs"),
    OUTPUT_DIR,

    # DGCNN robustness evaluation
    ROBUST_DIR,
    FIG_DIR,
    METRICS_DIR,
    CM_DIR,
    REPORT_DIR,
    CKPT_DIR,
]

for directory in ALL_DIRS:
    os.makedirs(directory, exist_ok=True)

# --- Semantic classes -------------------------------------------------------
SEMANTIC_CLASS_NAMES = {
    0: "Background / Pot / Soil",
    1: "Stem",
    2: "Leaf",
}

NUM_CLASSES = len(SEMANTIC_CLASS_NAMES)
RANDOM_SEED = 42

# --- Verification -----------------------------------------------------------
print("=" * 70)
print("DIRECTORY SETUP COMPLETE")
print("=" * 70)

for directory in ALL_DIRS:
    print(f"[OK] {directory}")

print("\n" + "=" * 70)
print("FILES")
print("=" * 70)
print("GT PLY          :", PLY_PATH)
print("Semantic labels :", SEMANTIC_PATH)
print("Instance labels :", INSTANCE_PATH)
print("Confidence      :", CONFIDENCE_PATH)
print("Manifest        :", MANIFEST_PATH)

print("\n" + "=" * 70)
print("ROBUSTNESS OUTPUT")
print("=" * 70)
print("Robustness out  :", ROBUST_DIR)
print("Figures         :", FIG_DIR)
print("Metrics         :", METRICS_DIR)
print("Confusion mats  :", CM_DIR)
print("Reports         :", REPORT_DIR)
print("Checkpoints     :", CKPT_DIR)
print("=" * 70)

DIRECTORY SETUP COMPLETE
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/gt_reconstruction
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/noisy_reconstruction
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/noisy_reconstruction/degraded
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/outputs
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/outputs/Ribes_04
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/outputs/Ribes_04/robustness_eval_dgcnn
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/outputs/Ribes_04/robustness_eval_dgcnn/figures
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/outputs/Ribes_04/robustness_eval_dgcnn/tables
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/outputs/Ribes_04/robustness_eval_dgcnn/tables/confusion_matrices
[OK] /content/drive/MyDrive/segmentation_jetcobot_internship/outputs/Ribes

In [4]:
import numpy as np

N_POINTS_PER_SAMPLE = 4096
K_NEIGHBORS         = 20
USE_RGB_FEATURES    = True
USE_COLOR_AUGMENT   = True
ADDITIONAL_CHANNELS = 3 if USE_RGB_FEATURES else 0

EPOCHS                    = 40
BATCH_SIZE                = 8
LEARNING_RATE             = 1e-3
LR_DECAY_STEP             = 15
LR_DECAY_GAMMA            = 0.5
WEIGHT_DECAY               = 1e-4
N_TRAIN_SAMPLES_PER_EPOCH = 300
N_VAL_SAMPLES_PER_EPOCH   = 60
TRAIN_POINT_FRACTION       = 0.85
N_REPEATS_EVAL             = 4

np.random.seed(RANDOM_SEED)
print(f"DGCNN Config: k={K_NEIGHBORS}, points={N_POINTS_PER_SAMPLE}, color_augment={USE_COLOR_AUGMENT}")

DGCNN Config: k=20, points=4096, color_augment=True


In [5]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    print("⚠️ No GPU detected — training will be slow.")
else:
    print("Using GPU:", torch.cuda.get_device_name(0))

Using GPU: Tesla T4


In [6]:
import open3d as o3d

def load_point_cloud(path):
    pcd = o3d.io.read_point_cloud(path)
    pts = np.asarray(pcd.points, dtype=np.float64)
    cols = np.asarray(pcd.colors, dtype=np.float64) if pcd.has_colors() else None
    return pts, cols

def load_label_file(path, dtype=np.int64):
    return np.loadtxt(path, dtype=dtype).reshape(-1)

gt_points, gt_rgb = load_point_cloud(PLY_PATH)
gt_semantic   = load_label_file(SEMANTIC_PATH, dtype=np.int64)
gt_instance   = load_label_file(INSTANCE_PATH, dtype=np.int64)
gt_confidence = load_label_file(CONFIDENCE_PATH, dtype=np.float64)

N_GT = gt_points.shape[0]
GLOBAL_CENTER = gt_points.mean(axis=0)
bbox_min, bbox_max = gt_points.min(axis=0), gt_points.max(axis=0)
CHAR_SIZE = float(np.linalg.norm(bbox_max - bbox_min))

def normalize_xyz(points):
    return (points - GLOBAL_CENTER) / CHAR_SIZE

def make_features(rgb):
    if not USE_RGB_FEATURES:
        return None
    return rgb.astype(np.float64) if rgb is not None else np.zeros((0, 3))

print(f"Loaded {N_GT:,} points for plant {PLANT_NAME}.")

Loaded 1,190,366 points for plant Ribes_04.


In [7]:
import torch.nn as nn
import torch.nn.functional as F

def knn(x, k):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]
    return idx

def get_graph_feature(x, k=20, idx=None):
    batch_size = x.size(0)
    num_points = x.size(2)
    x = x.view(batch_size, -1, num_points)
    if idx is None:
        idx = knn(x, k=k)

    device = x.device
    idx_base = torch.arange(0, batch_size, device=device).view(-1, 1, 1) * num_points
    idx = idx + idx_base
    idx = idx.view(-1)

    num_dims = x.size(1)
    x = x.transpose(2, 1).contiguous()
    feature = x.view(batch_size * num_points, -1)[idx, :]
    feature = feature.view(batch_size, num_points, k, num_dims)
    x = x.view(batch_size, num_points, 1, num_dims).repeat(1, 1, k, 1)
    feature = torch.cat((feature - x, x), dim=3).permute(0, 3, 1, 2).contiguous()
    return feature

class DGCNNSemSeg(nn.Module):
    def __init__(self, num_classes=3, k=20, additional_channel=3):
        super().__init__()
        self.k = k
        in_channel = 3 + additional_channel

        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(64)
        self.bn4 = nn.BatchNorm2d(128)
        self.bn5 = nn.BatchNorm1d(1024)

        self.conv1 = nn.Sequential(nn.Conv2d(in_channel * 2, 64, kernel_size=1, bias=False), self.bn1, nn.LeakyReLU(0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(64 * 2, 64, kernel_size=1, bias=False), self.bn2, nn.LeakyReLU(0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(64 * 2, 64, kernel_size=1, bias=False), self.bn3, nn.LeakyReLU(0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(64 * 2, 128, kernel_size=1, bias=False), self.bn4, nn.LeakyReLU(0.2))
        self.conv5 = nn.Sequential(nn.Conv1d(320, 1024, kernel_size=1, bias=False), self.bn5, nn.LeakyReLU(0.2))

        self.conv6 = nn.Sequential(nn.Conv1d(1344, 512, kernel_size=1, bias=False), nn.BatchNorm1d(512), nn.LeakyReLU(0.2))
        self.conv7 = nn.Sequential(nn.Conv1d(512, 256, kernel_size=1, bias=False), nn.BatchNorm1d(256), nn.LeakyReLU(0.2))
        self.dp1 = nn.Dropout(p=0.5)
        self.conv8 = nn.Conv1d(256, num_classes, kernel_size=1)

    def forward(self, xyz, features=None):
        if features is not None:
            x = torch.cat([xyz, features], dim=1)
        else:
            x = xyz
        batch_size = x.size(0)
        num_points = x.size(2)

        x1 = get_graph_feature(x, k=self.k)
        x1 = self.conv1(x1).max(dim=-1)[0]

        x2 = get_graph_feature(x1, k=self.k)
        x2 = self.conv2(x2).max(dim=-1)[0]

        x3 = get_graph_feature(x2, k=self.k)
        x3 = self.conv3(x3).max(dim=-1)[0]

        x4 = get_graph_feature(x3, k=self.k)
        x4 = self.conv4(x4).max(dim=-1)[0]

        x_concat = torch.cat((x1, x2, x3, x4), dim=1)
        x5 = self.conv5(x_concat)
        x5 = x5.max(dim=-1, keepdim=True)[0].repeat(1, 1, num_points)

        x_full = torch.cat((x_concat, x5), dim=1)
        x = self.conv6(x_full)
        x = self.conv7(x)
        x = self.dp1(x)
        x = self.conv8(x)
        return F.log_softmax(x, dim=1)

print('DGCNN defined.')

DGCNN defined.


In [8]:
from torch.utils.data import Dataset, DataLoader

class PlantDataset(Dataset):
    def __init__(self, points, features, labels, n_samples, n_points_per_sample=4096, augment_color=False):
        self.points = points.astype(np.float32)
        self.features = features.astype(np.float32) if features is not None else None
        self.labels = labels.astype(np.int64)
        self.n_samples = n_samples
        self.n_points = n_points_per_sample
        self.augment_color = augment_color
        self.N = points.shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        center_idx = np.random.randint(0, self.N)
        center_pt = self.points[center_idx]
        dists = np.sum((self.points - center_pt) ** 2, axis=1)
        crop_indices = np.argpartition(dists, min(self.n_points, self.N - 1))[:self.n_points]
        if len(crop_indices) < self.n_points:
            crop_indices = np.random.choice(crop_indices, self.n_points, replace=True)

        pts = self.points[crop_indices].copy()
        lbls = self.labels[crop_indices].copy()
        feats = self.features[crop_indices].copy() if self.features is not None else np.zeros((self.n_points, 0), dtype=np.float32)

        theta = np.random.uniform(0, 2 * np.pi)
        rot = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]], dtype=np.float32)
        pts = pts @ rot.T

        if self.augment_color and feats.shape[1] >= 3:
            gain = np.random.uniform(0.7, 1.3)
            bias = np.random.uniform(-0.15, 0.15)
            feats[:, :3] = np.clip(feats[:, :3] * gain + bias, 0.0, 1.0)

        return (
            torch.from_numpy(pts.T).float(),
            torch.from_numpy(feats.T).float() if feats.shape[1] > 0 else torch.zeros((0, self.n_points)).float(),
            torch.from_numpy(lbls).long()
        )

np.random.seed(RANDOM_SEED)
rand_perm = np.random.permutation(N_GT)
n_train = int(TRAIN_POINT_FRACTION * N_GT)
train_idx, val_idx = rand_perm[:n_train], rand_perm[n_train:]

gt_xyz_norm = normalize_xyz(gt_points)
gt_feat = make_features(gt_rgb)

train_dataset = PlantDataset(gt_xyz_norm[train_idx], gt_feat[train_idx] if gt_feat is not None else None, gt_semantic[train_idx], N_TRAIN_SAMPLES_PER_EPOCH, N_POINTS_PER_SAMPLE, augment_color=USE_COLOR_AUGMENT)
val_dataset   = PlantDataset(gt_xyz_norm[val_idx], gt_feat[val_idx] if gt_feat is not None else None, gt_semantic[val_idx], N_VAL_SAMPLES_PER_EPOCH, N_POINTS_PER_SAMPLE, augment_color=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [9]:
import time
from sklearn.metrics import confusion_matrix

class_counts = np.bincount(gt_semantic, minlength=NUM_CLASSES)
class_weights = 1.0 / np.sqrt(np.clip(class_counts, 1e-6, None))
class_weights = class_weights / class_weights.sum() * NUM_CLASSES
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

model = DGCNNSemSeg(num_classes=NUM_CLASSES, k=K_NEIGHBORS, additional_channel=ADDITIONAL_CHANNELS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_DECAY_STEP, gamma=LR_DECAY_GAMMA)
criterion = nn.NLLLoss(weight=class_weights_t)

best_val_miou = 0.0
best_ckpt_path = os.path.join(CKPT_DIR, "dgcnn_best.pth")

print('Training DGCNN...')
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0
    for pts, feats, lbls in train_loader:
        pts, lbls = pts.to(DEVICE), lbls.to(DEVICE)
        feats = feats.to(DEVICE) if feats.shape[1] > 0 else None
        optimizer.zero_grad()
        log_probs = model(pts, feats)
        loss = criterion(log_probs, lbls)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
    scheduler.step()

    model.eval()
    total_val_loss = 0.0
    val_preds, val_targets = [], []
    with torch.no_grad():
        for pts, feats, lbls in val_loader:
            pts, lbls = pts.to(DEVICE), lbls.to(DEVICE)
            feats = feats.to(DEVICE) if feats.shape[1] > 0 else None
            log_probs = model(pts, feats)
            loss = criterion(log_probs, lbls)
            total_val_loss += loss.item()
            preds = log_probs.argmax(dim=1).cpu().numpy().ravel()
            val_preds.extend(preds)
            val_targets.extend(lbls.cpu().numpy().ravel())

    cm = confusion_matrix(val_targets, val_preds, labels=list(range(NUM_CLASSES)))
    ious = [cm[c, c] / (cm[c, :].sum() + cm[:, c].sum() - cm[c, c]) for c in range(1, NUM_CLASSES)]
    val_miou = float(np.mean(ious))

    if val_miou > best_val_miou:
        best_val_miou = val_miou
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_miou': val_miou}, best_ckpt_path)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {total_train_loss/len(train_loader):.4f} | Val Loss: {total_val_loss/len(val_loader):.4f} | Val mIoU: {val_miou:.4f} | Best: {best_val_miou:.4f}")

print(f"Training complete in {(time.time() - start_time)/60:.2f} mins.")

Training DGCNN...
Epoch 01/40 | Train Loss: 0.7309 | Val Loss: 0.4634 | Val mIoU: 0.4686 | Best: 0.4686
Epoch 05/40 | Train Loss: 0.3734 | Val Loss: 0.3917 | Val mIoU: 0.6120 | Best: 0.6120
Epoch 10/40 | Train Loss: 0.3460 | Val Loss: 0.2989 | Val mIoU: 0.7710 | Best: 0.7710
Epoch 15/40 | Train Loss: 0.2464 | Val Loss: 0.2909 | Val mIoU: 0.6649 | Best: 0.8228
Epoch 20/40 | Train Loss: 0.1942 | Val Loss: 0.2667 | Val mIoU: 0.7499 | Best: 0.8228
Epoch 25/40 | Train Loss: 0.2488 | Val Loss: 0.2939 | Val mIoU: 0.7607 | Best: 0.8228
Epoch 30/40 | Train Loss: 0.1761 | Val Loss: 0.1913 | Val mIoU: 0.7804 | Best: 0.8228
Epoch 35/40 | Train Loss: 0.1907 | Val Loss: 0.2170 | Val mIoU: 0.7594 | Best: 0.8504
Epoch 40/40 | Train Loss: 0.1381 | Val Loss: 0.1955 | Val mIoU: 0.7865 | Best: 0.8504
Training complete in 23.53 mins.


In [10]:
def predict_full_cloud_dgcnn(model, points, features, chunk_size=4096, n_repeats=4):
    model.eval()
    N = points.shape[0]
    accum_probs = np.zeros((N, NUM_CLASSES), dtype=np.float32)
    accum_counts = np.zeros(N, dtype=np.float32)
    with torch.no_grad():
        for rep in range(n_repeats):
            perm = np.random.permutation(N)
            for i in range(0, N, chunk_size):
                chunk_idx = perm[i:i + chunk_size]
                if len(chunk_idx) < chunk_size:
                    pad = np.random.choice(chunk_idx, chunk_size - len(chunk_idx), replace=True)
                    eval_idx = np.concatenate([chunk_idx, pad])
                else:
                    eval_idx = chunk_idx
                pts_t = torch.from_numpy(points[eval_idx].T[None, ...]).float().to(DEVICE)
                feats_t = torch.from_numpy(features[eval_idx].T[None, ...]).float().to(DEVICE) if features is not None else None
                log_probs = model(pts_t, feats_t)
                probs = torch.exp(log_probs).squeeze(0).cpu().numpy().T
                accum_probs[eval_idx] += probs
                accum_counts[eval_idx] += 1.0
    accum_counts = np.maximum(accum_counts, 1.0)[:, None]
    avg_probs = accum_probs / accum_counts
    preds = avg_probs.argmax(axis=1)
    mean_conf = float(avg_probs.max(axis=1).mean())
    return preds, avg_probs, mean_conf

def compute_metrics(gt_labels, pred_labels):
    cm = confusion_matrix(gt_labels, pred_labels, labels=list(range(NUM_CLASSES)))
    iou_per_class, prec_per_class, rec_per_class, f1_per_class = [], [], [], []
    for c in range(NUM_CLASSES):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp
        fn = cm[c, :].sum() - tp
        union = tp + fp + fn
        iou = tp / union if union > 0 else np.nan
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        iou_per_class.append(iou)
        prec_per_class.append(prec)
        rec_per_class.append(rec)
        f1_per_class.append(f1)
    valid_ious = [iou_per_class[c] for c in range(1, NUM_CLASSES) if not np.isnan(iou_per_class[c])]
    miou = float(np.mean(valid_ious)) if valid_ious else 0.0
    overall_acc = float((gt_labels == pred_labels).mean())
    return {'miou': miou, 'overall_accuracy': overall_acc, 'iou_per_class': iou_per_class, 'precision_per_class': prec_per_class, 'recall_per_class': rec_per_class, 'f1_per_class': f1_per_class, 'confusion_matrix': cm}

In [11]:
import json

ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

gt_xyz_norm_full = normalize_xyz(gt_points)
gt_features_full = make_features(gt_rgb)

t_start = time.time()
gt_preds, _, gt_mean_conf = predict_full_cloud_dgcnn(model, gt_xyz_norm_full, gt_features_full, chunk_size=N_POINTS_PER_SAMPLE, n_repeats=N_REPEATS_EVAL)
gt_eval_time = time.time() - t_start

gt_metrics = compute_metrics(gt_semantic, gt_preds)
gt_metrics['mean_confidence'] = gt_mean_conf
gt_metrics['inference_time_s'] = gt_eval_time

with open(os.path.join(METRICS_DIR, 'gt_baseline_metrics.json'), 'w') as f:
    json.dump({k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in gt_metrics.items()}, f, indent=2)

np.save(os.path.join(CM_DIR, 'GT_confusion_matrix.npy'), gt_metrics['confusion_matrix'])
print(f"✅ GT Baseline Evaluated & Saved | mIoU: {gt_metrics['miou']:.4f}")

✅ GT Baseline Evaluated & Saved | mIoU: 0.6205


In [18]:
import glob
import os
import time
import numpy as np
import open3d as o3d
import pandas as pd
from tqdm import tqdm


# --- Helper function to locate the exact .ply file for any manifest format ---
def resolve_ply_path(row, plant_name, base_dir, noisy_dir, degraded_dir):
  candidates = []

  # 1. From 'output_dir' column (standard in PLANesT-3D degradation pipeline)
  if "output_dir" in row and pd.notna(row["output_dir"]):
    out_dir = str(row["output_dir"]).strip()
    candidates.extend([
        out_dir,
        os.path.join(base_dir, out_dir),
        os.path.join(noisy_dir, out_dir),
        os.path.join(degraded_dir, out_dir),
    ])

  # 2. Direct folder lookup: degraded_dir / category / level
  cat = str(row.get("category", "")).strip()
  lvl = str(row.get("level", "")).strip()
  if cat and lvl:
    candidates.extend(
        [os.path.join(degraded_dir, cat, lvl), os.path.join(noisy_dir, cat, lvl)]
    )

  # Check candidate paths
  for path in candidates:
    if os.path.isfile(path) and path.lower().endswith(".ply"):
      return path
    if os.path.isdir(path):
      exact_ply = os.path.join(path, f"{plant_name}.ply")
      if os.path.isfile(exact_ply):
        return exact_ply
      ply_files = glob.glob(os.path.join(path, "*.ply"))
      if len(ply_files) > 0:
        return ply_files[0]

  raise FileNotFoundError(
      f"Could not find .ply file for category '{cat}', level '{lvl}' in row:"
      f" {dict(row)}"
  )


# --- Load Manifest ---
manifest_df = pd.read_csv(MANIFEST_PATH)
print("Detected manifest columns:", list(manifest_df.columns))

results_list = []
summary_csv_path = os.path.join(METRICS_DIR, "summary_degradation_table.csv")
variant_csv_path = os.path.join(METRICS_DIR, "variant_metrics.csv")

# Pre-build Ground Truth KDTree for fast 1-NN label mapping
pcd_gt = o3d.geometry.PointCloud()
pcd_gt.points = o3d.utility.Vector3dVector(gt_points)
kdtree = o3d.geometry.KDTreeFlann(pcd_gt)

print(f"\n🚀 Starting Evaluation on {len(manifest_df)} noisy variants...\n")

for idx, row in tqdm(manifest_df.iterrows(), total=len(manifest_df)):
  category = str(row["category"])
  level = str(row["level"])

  # Locate the degraded .ply file correctly
  variant_ply = resolve_ply_path(
      row, PLANT_NAME, BASE_DIR, NOISY_DIR, DEGRADED_DIR
  )

  # Load point cloud
  v_pts, v_cols = load_point_cloud(variant_ply)

  if v_pts is None or len(v_pts) == 0:
    raise ValueError(
        f"Loaded 0 points from '{variant_ply}'. File may be corrupt or missing."
    )

  v_xyz_norm = normalize_xyz(v_pts)
  v_feats = make_features(v_cols)

  # 1-NN ground truth transfer from clean GT cloud
  v_gt_labels = np.zeros(v_pts.shape[0], dtype=np.int64)
  for i in range(v_pts.shape[0]):
    _, idx_nn, _ = kdtree.search_knn_vector_3d(v_pts[i], 1)
    v_gt_labels[i] = gt_semantic[idx_nn[0]]

  # Model Prediction
  t_start = time.time()
  v_preds, _, v_conf = predict_full_cloud_dgcnn(
      model,
      v_xyz_norm,
      v_feats,
      chunk_size=N_POINTS_PER_SAMPLE,
      n_repeats=N_REPEATS_EVAL,
  )
  v_time = time.time() - t_start

  # Compute Metrics
  v_m = compute_metrics(v_gt_labels, v_preds)
  delta_miou = gt_metrics["miou"] - v_m["miou"]
  degradation_pct = (
      (delta_miou / gt_metrics["miou"]) * 100.0 if gt_metrics["miou"] > 0 else 0.0
  )

  res = {
      "model": "DGCNN ( baseline )",
      "category": category,
      "level": level,
      "gt_miou": gt_metrics["miou"],
      "miou": v_m["miou"],
      "delta_miou": delta_miou,
      "degradation_pct": degradation_pct,
      "overall_accuracy": v_m["overall_accuracy"],
      "mean_confidence": v_conf,
      "inference_time_s": v_time,
      "n_points": v_pts.shape[0],
  }
  results_list.append(res)

  # Save confusion matrix array
  cm_filename = f"{category}_{level}_confusion_matrix.npy".replace("/", "_")
  np.save(os.path.join(CM_DIR, cm_filename), v_m["confusion_matrix"])

  # --- INCREMENTAL SAVING TO DISK AFTER EVERY SINGLE VARIANT ---
  df_temp = pd.DataFrame(results_list)
  df_temp.to_csv(variant_csv_path, index=False)

  summary_df_temp = pd.DataFrame({
      "Model": df_temp["model"],
      "Category": df_temp["category"],
      "Level": df_temp["level"],
      "GT mIoU": np.round(df_temp["gt_miou"], 4),
      "Noisy mIoU": np.round(df_temp["miou"], 4),
      "ΔmIoU": np.round(df_temp["delta_miou"], 4),
      "Degradation %": np.round(df_temp["degradation_pct"], 2),
  })
  summary_df_temp.to_csv(summary_csv_path, index=False)

print(
    "✅ Incremental evaluation finished! Real metrics successfully computed"
    " and saved."
)

Detected manifest columns: ['category', 'level', 'params', 'n_points', 'pct_of_clean', 'output_dir']

🚀 Starting Evaluation on 21 noisy variants...



100%|██████████| 21/21 [16:44<00:00, 47.84s/it]

✅ Incremental evaluation finished! Real metrics successfully computed and saved.


In [19]:
summary_df = pd.read_csv(summary_csv_path)
display(summary_df)

,Model,Category,Level,GT mIoU,Noisy mIoU,ΔmIoU,Degradation %
0,DGCNN ( baseline ),density,100,0.6205,0.6191,0.0014,0.23
1,DGCNN ( baseline ),density,75,0.6205,0.6217,-0.0012,-0.20
2,DGCNN ( baseline ),density,50,0.6205,0.6212,-0.0007,-0.12
3,DGCNN ( baseline ),density,25,0.6205,0.6156,0.0048,0.78
4,DGCNN ( baseline ),density,10,0.6205,0.6128,0.0077,1.24
5,DGCNN ( baseline ),coordinate_noise,low,0.6205,0.6201,0.0004,0.06
6,DGCNN ( baseline ),coordinate_noise,medium,0.6205,0.6134,0.0071,1.14
7,DGCNN ( baseline ),coordinate_noise,high,0.6205,0.5990,0.0215,3.47
8,DGCNN ( baseline ),missing_points,10,0.6205,0.6003,0.0202,3.26
9,DGCNN ( baseline ),missing_points,30,0.6205,0.6046,0.0159,2.56


In [20]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 8))
for cat in summary_df['Category'].unique():
    sub = summary_df[summary_df['Category'] == cat]
    plt.plot(sub['Level'], sub['Noisy mIoU'], marker='o', linewidth=2.5, label=cat)

plt.axhline(gt_metrics['miou'], color='red', linestyle='--', label='GT Baseline')
plt.title('DGCNN: Robustness to Reconstruction Degradation (mIoU vs Severity)', fontsize=14, fontweight='bold')
plt.xlabel('Degradation Severity / Level', fontsize=12)
plt.ylabel('mIoU', fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_miou_by_category_dgcnn.png'), dpi=300)
plt.close()

plt.figure(figsize=(10, 8))
pivot_df = summary_df.pivot(index='Category', columns='Level', values='Degradation %')
sns.heatmap(pivot_df, annot=True, fmt='.1f', cmap='YlOrRd', cbar_kws={'label': 'Degradation %'})
plt.title('DGCNN: Degradation % Heatmap across Categories & Levels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_degradation_pct_heatmap_dgcnn.png'), dpi=300)
plt.close()
os.system('sync')
print('✅ Figures saved to disk.')

✅ Figures saved to disk.


In [21]:
report_path = os.path.join(REPORT_DIR, 'degradation_robustness_report.md')
report_text = f"""# DGCNN Segmentation Robustness Report — Plant: `{PLANT_NAME}`

## 1. Baseline Performance
- **Model:** Dynamic Graph CNN (DGCNN, k={K_NEIGHBORS})
- **GT Baseline mIoU:** `{gt_metrics['miou']:.4f}`
- **GT Overall Accuracy:** `{gt_metrics['overall_accuracy']:.4f}`

## 2. Summary Degradation Table
| Category | Level | Noisy mIoU | ΔmIoU | Degradation % |
|---|---|---|---|---|
"""
for _, r in summary_df.iterrows():
    report_text += f"| {r['Category']} | {r['Level']} | {r['Noisy mIoU']:.4f} | {r['ΔmIoU']:.4f} | {r['Degradation %']:.2f}% |\n"

with open(report_path, 'w') as f:
    f.write(report_text)

print(f"✅ Markdown report saved at: {report_path}")

✅ Markdown report saved at: /content/drive/MyDrive/segmentation_jetcobot_internship/outputs/Ribes_04/robustness_eval_dgcnn/report/degradation_robustness_report.md
